# 权重初始化

**面试回答：**初始化应让前向激活和反向梯度的方差跨层稳定；Xavier 对近线性/tanh，He 对 ReLU。

## 真实案例

质量检测网络接收 100 条传感器向量，比较零、过大、Xavier 与 He 初始化。

In [1]:
import numpy as np  # 导入 NumPy 模拟网络信号。
rng=np.random.default_rng(4)  # 固定随机种子。
x=rng.normal(size=(100,16))  # 构造设备传感器批次。
print('传感器批次形状=',x.shape,'方差=',round(float(x.var()),3))  # 输出输入统计。
print('教学任务：观察三层 ReLU 激活方差。')  # 说明任务。

传感器批次形状= (100, 16) 方差= 1.02
教学任务：观察三层 ReLU 激活方差。


## Baseline / 基线

零初始化会让所有隐藏单元对称。

In [2]:
zero=np.zeros((16,16))  # 构造零权重基线。
zero_out=np.maximum(0,x@zero)  # 计算零初始化 ReLU 输出。
print('零初始化激活方差=',float(zero_out.var()))  # 输出对称失败。

零初始化激活方差= 0.0


In [3]:
def propagate(scale):  # 定义三层 ReLU 方差传播实验。
    h=x.copy()  # 复制输入激活。
    values=[]  # 保存每层方差。
    for layer in range(3):  # 构造三层线性加 ReLU。
        w=rng.normal(0,scale,size=(16,16))  # 按指定尺度初始化权重。
        h=np.maximum(0,h@w)  # 计算当前层 ReLU 激活。
        values.append(float(h.var()))  # 记录当前激活方差。
    return values  # 返回方差轨迹。
tiny=propagate(.01)  # 运行过小初始化。
large=propagate(1.0)  # 运行过大初始化。
he=propagate(np.sqrt(2/16))  # 运行 He 初始化。
print('过小/过大/He方差=',np.round(tiny,4),np.round(large,4),np.round(he,4))  # 输出中间量。

过小/过大/He方差= [0.0004 0.     0.    ] [  5.5452  51.4548 324.5101] [0.7119 0.864  0.8404]


## 结果解读

过小方差消失，过大方差爆炸；He 用 fan-in 补偿 ReLU 丢失的一半信号。

In [4]:
print('方案 | 第三层激活方差')  # 输出结果表头。
print('零',float(zero_out.var()))  # 输出零初始化。
print('过小',round(tiny[-1],6))  # 输出消失现象。
print('He',round(he[-1],6))  # 输出稳定方案。
print('生产差距：应同时检查梯度、残差、归一化和预训练权重加载。')  # 说明边界。

方案 | 第三层激活方差
零 0.0
过小 0.0
He 0.840365
生产差距：应同时检查梯度、残差、归一化和预训练权重加载。


## 失败案例与修复

给所有单元同一个零权重会造成对称；修复是随机但方差受控的初始化。

In [5]:
print('失败零方差=',float(zero_out.var()))  # 输出对称失败证据。
print('修复He首层方差=',round(he[0],4))  # 输出受控随机初始化证据。
print('初始化不替代数据缩放和学习率选择。')  # 说明边界。
print('随机种子应随实验记录。')  # 说明可复现性。

失败零方差= 0.0
修复He首层方差= 0.7119
初始化不替代数据缩放和学习率选择。
随机种子应随实验记录。


In [6]:
assert x.shape[0]>=5  # 保护样本数。
assert zero_out.var()==0  # 保护零初始化失败。
assert tiny[-1]<tiny[0]  # 保护过小方差消失。
assert large[-1]>large[0]  # 保护过大方差放大。